# Fine-tune PhoBERT trên Google Colab

Notebook này fine-tune PhoBERT để phân loại URL thành benign hoặc phishing bằng ba file split của PhishVN.

1. Trong Colab, chọn Runtime > Change runtime type > T4 GPU (hoặc GPU khả dụng).
2. Chạy lần lượt các cell từ trên xuống dưới.
3. Ở cell dữ liệu, tải lên url_train.csv, url_val.csv và url_test.csv nếu các file chưa có trong Google Drive.
4. Mô hình và metrics sẽ được lưu trong MyDrive/phoBERT_output.

Các thông số train có thể thay đổi trong cell Cấu hình bên dưới.

In [ ]:
%pip install -q "transformers>=4.40,<5" accelerate sentencepiece scikit-learn pandas

In [ ]:
import shutil
from pathlib import Path

import torch
from google.colab import drive, files

print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU chưa được bật. Vào Runtime > Change runtime type > chọn GPU.")

drive.mount("/content/drive")

required_files = ["url_train.csv", "url_val.csv", "url_test.csv"]
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/PhishVN/data/splits")
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_files = [name for name in required_files if not (DRIVE_DATA_DIR / name).is_file()]
if missing_files:
    print("Tải lên các file CSV split. Nên chọn đủ cả 3 file:")
    uploaded = files.upload()
    for name in required_files:
        if name in uploaded:
            (DRIVE_DATA_DIR / name).write_bytes(uploaded[name])
    still_missing = [name for name in required_files if not (DRIVE_DATA_DIR / name).is_file()]
    if still_missing:
        raise FileNotFoundError(f"Chưa có các file: {still_missing}")

DATA_DIR = Path("/content/phobert_data/splits")
DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in required_files:
    shutil.copy2(DRIVE_DATA_DIR / name, DATA_DIR / name)

print("Dữ liệu sẵn sàng tại:", DATA_DIR)

In [ ]:
# Cấu hình model và training: sửa các giá trị tại cell này trước khi chạy train.
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 128
EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LR_SCHEDULER_TYPE = "linear"
CLASSIFIER_DROPOUT = 0.1
HIDDEN_DROPOUT_PROB = 0.1
ATTENTION_DROPOUT_PROB = 0.1
GRADIENT_CHECKPOINTING = False
FP16 = torch.cuda.is_available()
LOGGING_STEPS = 100
SEED = 42

MODEL_OUT = Path("/content/drive/MyDrive/phoBERT_output/model")
METRICS_OUT = Path("/content/drive/MyDrive/phoBERT_output/metrics.json")
CHECKPOINT_DIR = Path("/content/phobert_checkpoints")

print(f"Model: {MODEL_NAME}")
print(f"Sequence length: {MAX_LENGTH}; epochs: {EPOCHS}; fp16: {FP16}")

In [ ]:
import inspect
import json

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

LABEL_TO_ID = {"benign": 0, "phishing": 1, "0": 0, "1": 1}

def load_split(split_name, filename):
    path = DATA_DIR / filename
    frame = pd.read_csv(path, usecols=["url", "label"])
    frame["url"] = frame["url"].fillna("").astype("string").str.strip()
    raw_labels = frame["label"].astype("string").str.strip().str.lower()

    unknown = sorted(set(raw_labels.dropna()) - set(LABEL_TO_ID))
    if unknown:
        raise ValueError(f"Label không hợp lệ trong {filename}: {unknown}")

    frame["label_id"] = raw_labels.map(LABEL_TO_ID)
    frame = frame.loc[frame["url"].ne("") & frame["label_id"].notna()].copy()
    if frame.empty:
        raise ValueError(f"Tập {split_name} không có URL hợp lệ.")
    labels = frame["label_id"].astype("int64").tolist()
    if len(set(labels)) < 2:
        raise ValueError(f"Tập {split_name} phải có cả benign và phishing.")
    return frame["url"].astype(str).tolist(), labels

class URLDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {
            key: torch.tensor(values[index], dtype=torch.long)
            for key, values in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

def make_dataset(tokenizer, texts, labels):
    encodings = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )
    return URLDataset(encodings, labels)

split_files = {
    "train": "url_train.csv",
    "validation": "url_val.csv",
    "test": "url_test.csv",
}
datasets = {}
row_counts = {}
for split_name, filename in split_files.items():
    texts, labels = load_split(split_name, filename)
    datasets[split_name] = (texts, labels)
    row_counts[split_name] = len(labels)
    print(f"{split_name}: {len(labels):,} URLs")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
for split_name, (texts, labels) in datasets.items():
    datasets[split_name] = make_dataset(tokenizer, texts, labels)

print("Tokenization hoàn tất.")

In [ ]:
def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits)
    labels = np.asarray(eval_prediction.label_ids)

    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    predicted = probabilities.argmax(axis=1)

    return {
        "accuracy": float(accuracy_score(labels, predicted)),
        "precision": float(precision_score(labels, predicted, pos_label=1, zero_division=0)),
        "recall": float(recall_score(labels, predicted, pos_label=1, zero_division=0)),
        "f1": float(f1_score(labels, predicted, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, probabilities[:, 1])),
    }

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "benign", 1: "phishing"},
    label2id={"benign": 0, "phishing": 1},
    classifier_dropout=CLASSIFIER_DROPOUT,
    hidden_dropout_prob=HIDDEN_DROPOUT_PROB,
    attention_probs_dropout_prob=ATTENTION_DROPOUT_PROB,
)

In [ ]:
eval_argument = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments).parameters
    else "evaluation_strategy"
)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    fp16=FP16,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    dataloader_num_workers=0,
    **{eval_argument: "epoch"},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print(f"Bắt đầu train trên GPU: {torch.cuda.is_available()}")
train_result = trainer.train()

In [ ]:
validation_metrics = trainer.evaluate(
    eval_dataset=datasets["validation"],
    metric_key_prefix="validation",
)
test_metrics = trainer.evaluate(
    eval_dataset=datasets["test"],
    metric_key_prefix="test",
)

MODEL_OUT.mkdir(parents=True, exist_ok=True)
METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_OUT))
tokenizer.save_pretrained(str(MODEL_OUT))

results = {
    "model_name": MODEL_NAME,
    "label_to_id": {"benign": 0, "phishing": 1},
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "fp16": FP16,
    "seed": SEED,
    "rows": row_counts,
    "train": train_result.metrics,
    "validation": validation_metrics,
    "test": test_metrics,
}
METRICS_OUT.write_text(
    json.dumps(results, indent=2, ensure_ascii=False, default=float),
    encoding="utf-8",
)

print("Model và tokenizer đã lưu tại:", MODEL_OUT)
print("Metrics đã lưu tại:", METRICS_OUT)
print("Test metrics:", test_metrics)